# How many hashes until two agree?
## The Birthday Paradox in Truncated SHA-256
### *A study of first-collision waiting times*

Imagine standing in a classroom doorway are recording each student's birthday as they arrive to class (excluding leap year birthdays).

$$
\begin{align*}
P(\text{no collisions among 1 person}) &= 1\\
P(\text{no collisions among $2$ people}) &= \left(1-\frac{1}{365}\right) = 0.99726\\
P(\text{no collisions among $3$ people}) &= \left(1-\frac{1}{365}\right)\left(1-\frac{2}{365}\right) = 0.991796\\ 
P(\text{no collisions among $4$ people}) &= \left(1-\frac{1}{365}\right)\left(1-\frac{2}{365}\right)\left(1-\frac{3}{365}\right)=0.983644\\
&\ \ \vdots  \\ %lines up spacing
\\  
\end{align*}
$$
For $k$ independently modeled birthdays uniformly distributed over $365$ days: 
$$
\begin{align*} \\
P(\text{no collisions among $k$}) &= \prod_{i=1}^{k-1} \left( 1 - \frac{i}{365} \right) \\
P(\text{at least 1 collision}) &= 1 - \prod_{i=1}^{k-1} \left( 1 - \frac{i}{365} \right) 
\end{align*}
$$


For $k = 23$, $\text{[Paar \S11.2.3 p. 342-343]}$
$$
\begin{align*}\\
P(\text{at least 1 collision}) &= 1- \prod_{i=1}^{22} \left( 1 - \frac{i}{365} \right)  \approx0.507297 \qquad 
\end{align*}
$$


>[Paar] C. Paar, J. Pelzl, and T. Güneysu, *Understanding Cryptography: From Established Symmetric and Asymmetric Ciphers to Post-Quantum Algorithms*, 2nd ed. Berlin, Heidelberg: Springer, 2024, doi: 10.1007/978-3-662-69007-9.

Here distinct messages replace human subjects. The first $b$ bits of the SHA-256 digests we examine takes the place of birthdays. 

A hash function takes an input message and generates a "label" or "fingerprint" of the message called a digest.

Hash functions are deterministic, meaning the same input message will always produce the same output digest. 

Rather than 365 possible birthdays, we have $M=2^b$ possible digests. 

$$
P(\text{no collisions among $k$}) = \prod_{i=1}^{k-1} \left( 1 - \frac{i}{M} \right)
$$

We count one number per trial: $T$, the number of messages hashed including the repeated hash. This is the hashes-until-collision value. 

A trial that ends at $T=6$ has given $6$ distinct messages to the hash function and seen only 5 distinct labels, implying the 6th distinct message produced a hash collision. 

Start with the one-byte ($b=8$ bits) example. To generate another dataset, change the three settings under Run the Experiment, then rerun all the cells with a Python kernel that has  $\texttt{NumPy}$ installed.

In [1]:
import csv
import hashlib
from pathlib import Path

import numpy as np

# Giving Each Message a Unique Name

A trial has identifier $r$, and its messages are numbered $i=1,2,3,\ldots$. 

Store each number in $8$ bytes, most significant byte first, and join them. 

Use a fresh $r$ for each trial to keep the message sets disjoint. 

In [2]:
U64_MAX = 2**64 - 1  # Maximum value of a 64-bit unsigned integer

In [3]:
def integer_in_range(value: int, name: str, low: int, high: int | None = None) -> int:
    """Validates that the given value is an integer within the specified range.
    Args:
        value (int): The value to validate.
        name (str): The name of the variable being validated (used in error messages).
        low (int): The lower bound of the valid range (inclusive).
        high (int | None): The upper bound of the valid range (inclusive). If None, no upper bound is enforced.
    Raises:
        TypeError: If the value is not an integer.
        ValueError: If the value is outside the specified range.
    Returns:
        int: The validated integer value.
    """
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, int):
        raise TypeError(f"{name} must be an integer")

    if value < low or (high is not None and value > high):
        raise ValueError(f"{name} must be in the range [{low}, {high}]")

    return value


def encode(trial_id: int, counter: int) -> bytes:
    """Encode (trial_id, counter) as two unsigned 64-bit integers in big-endian order
    Args:
        trial_id (int): The trial ID to encode.
        counter (int): The counter value to encode.
    Raises:
        TypeError: If trial_id or counter is not an integer.
        ValueError: If trial_id or counter is outside the range of a 64-bit unsigned integer.
    Returns:
        bytes: A byte string representing the encoded (trial_id, counter) pair.
    """
    trial_id = integer_in_range(trial_id, "trial_id", 0, U64_MAX)
    counter = integer_in_range(counter, "counter", 0, U64_MAX)

    return trial_id.to_bytes(8, byteorder="big") + counter.to_bytes(8, byteorder="big")

# Turn a Digest into a Label
SHA-256 returns $256$ bits. 

We keep the leading $b$ bits and read them as an integer from $0$ to $2^b-1$. 

$8$ bits ($1$ byte) gives $256$ labels and $16$ bits ($2$ bytes) gives $65,536$. The width is measured in bits $b$. 

Non-byte widths work as well. 

For a $12$-bit label, we read the first $16$ bits and discard the last $4$ bits. 

The digest is read as `eb29` and becomes `eb2`.  

In [4]:
def truncate_sha_256(message: bytes, bits: int) -> int:
    """Return the first leading 1-256 bits of the SHA-256 hash of the given message as a nonnegative integer.
    Args:
        message (bytes): The input message to hash.
        bits (int): The number of leading bits to return from the hash (1-256).
    Raises:
        ValueError: If bits is not in the range [1, 256].
    Returns:
        int: The first leading bits of the SHA-256 hash as a nonnegative integer.
    """
    if not isinstance(message, (bytes, bytearray)):
        raise TypeError("message must be bytes or bytearray")
        
    bits = integer_in_range(bits, "bits", 1, 256)
    byte_count = (
        bits + 7
    ) // 8  # Calculate the number of bytes needed to cover the specified bits
    leading_bytes = hashlib.sha256(message).digest()[:byte_count]
    unused_bits = 8 * byte_count - bits
    return int.from_bytes(leading_bytes, byteorder="big") >> unused_bits

# Following One Trial Through The Algorithm

We start with an empty dictionary that keeps track of draws and their unique labels. 

After we encode a unique message, we apply the hash function and take the prefix of the digest which becomes the label. 

Before we increment the draw counter, we ask if the label has already appeared in the dictionary and if so, return the collision. 

Otherwise record where exactly the unique label appears. 

With $M$ labels, the pigeonhole principle guarantees a repeat by draw $M+1$ when the counter can reach it. 

The $64$-bit counter begins at $1$ and can reach $2^{64}-1$. 

Small widths such as $8$, $12$, and $16$ make the experiment practical to compute. 

In [5]:
def collision_search(trial_id: int, bits: int, *, trace: bool = False) -> int:
    """Count draws through the first repeated prefix, optionally showing each draw.
    Args:
        trial_id (int): The ID of the trial.
        bits (int): The number of bits to consider for the prefix.
        trace (bool, optional): Whether to show each draw. Defaults to False.
    Raises:
        OverflowError: If the 64-bit counter is exhausted before finding a repeat.
    Returns:
        int: The number of draws until the first repeated prefix is found.
    """
    bits = integer_in_range(bits, "bits", 1, 256)
    last_counter = min(2**bits + 1, U64_MAX)
    first_seen_at: dict[int, int] = {}  # seen prefixes starts empty

    if trace:
        print("Draw Prefix Observation")

    for counter in range(1, last_counter + 1):
        message = encode(trial_id, counter)
        prefix = truncate_sha_256(message, bits)
        previous = first_seen_at.get(prefix)  # Check if we've seen this prefix before

        if trace:
            digits = (bits + 3) // 4
            label = format(prefix, f"0{digits}x")
            note = "new" if previous is None else f"matches draw {previous}"
            print(f"{counter:>4} {label:>6} {note}")

        if previous is not None:  # We have seen this prefix before (a collision)
            return counter
        first_seen_at[prefix] = counter # Else record the first occurrence of this prefix (a survival)

    raise OverflowError("Exhausted the 64-bit counter before finding a repeat.")  # violates experimental assumption (T <= 2^bits)

In [6]:
demo_T = collision_search(trial_id=82, bits=8, trace=True)
print(f"Recorded waiting time: T = {demo_T}")

Draw Prefix Observation
   1     c8 new
   2     15 new
   3     eb new
   4     6a new
   5     eb matches draw 3
Recorded waiting time: T = 5


# Run the Experiment

In [7]:
# Settings
# ========
BITS = 16
TRIALS = 900
MASTER_SEED = 37430101

In [8]:
def experiment_driver(trials: int, bits: int, master_seed: int) -> list[dict]:
    """Return one row per completed trial in collection order.
    Args:
        trials (int): The number of trials to run.
        bits (int): The number of bits to consider for the prefix.
        master_seed (int): The master seed for the random number generator.
    Raises:
        TypeError: If trials, bits, or master_seed is not an integer.
        ValueError: If trials is less than 1, bits is not in the range [1, 256], or master_seed is negative.
    Returns:
        list[dict]: A list of dictionaries, each representing a trial with its parameters and results.
    """
    trials = integer_in_range(trials, "trials", 1, 2**63)
    bits = integer_in_range(bits, "bits", 1, 256)
    master_seed = integer_in_range(master_seed, "master_seed", 0)

    rng = np.random.Generator(np.random.PCG64([master_seed, bits]))
    used_ids: set[int] = set()
    rows = []
    
    while len(rows) < trials:
        trial_id = int(rng.integers(0, 2**63, dtype=np.uint64))
        if trial_id in used_ids:
            continue
        used_ids.add(trial_id)
        rows.append(
            {
                "trial": len(rows) + 1,
                "trial_id": str(trial_id),
                "bits": bits,
                "master_seed": str(master_seed),
                "T": collision_search(trial_id, bits),
            }
        )
    return rows

In [9]:
rows = experiment_driver(TRIALS, BITS, MASTER_SEED)
waits = np.array([row["T"] for row in rows])

print(f"{len(rows):,} trials | {sum(row['T'] for row in rows):,} hash evaluations")
print(f"Sample mean = {waits.mean():.3f} | Sample median = {np.median(waits):g}")
print("First row:", rows[0])

900 trials | 284,233 hash evaluations
Sample mean = 315.814 | Sample median = 294
First row: {'trial': 1, 'trial_id': '3931139061786703017', 'bits': 16, 'master_seed': '37430101', 'T': 408}


# What does the Birthday Model Predict?

 Let $S(k)=P(T>k)$ define the probability that the first $k$ labels are all unique. 

 Given $k$ labels, the next draw has $M-k$ choices that let the trial survive, and 

$$
S(0) = 1, \qquad S(k + 1) = S(k)\left(1 - \frac{k}{M}\right), \qquad F(k) = P(T \le k) = 1 - S(k).
$$

For 16 bits, the first $k$ with $F(k)\ge 0.5$ (median) is 302. 

The exact model mean, $E[T]=\sum_{k=0}^{M}S(k)$, is about 321.52 hashes. 

The median answers “by when have half the trials stopped?” and the mean answers “how many hashes does a trial use on average?”

In [10]:
def collision_probability(k: int, outcomes: int) -> float:
    """Return the probability of at least one collision in k draws from outcomes possibilities.
    Args:
        k (int): The number of draws.
        outcomes (int): The number of possible outcomes.
    Raises:
        TypeError: If k or outcomes is not an integer.
        ValueError: If k is negative or greater than outcomes, or if outcomes is less than 1.
    Returns:
        float: The probability of at least one collision.
    """
    k = integer_in_range(k, "k", 0, outcomes)
    outcomes = integer_in_range(outcomes, "outcomes", 1)
    if k > outcomes:
        return 1.0
    used = np.arange(k, dtype=np.float64)
    log_survival = np.log1p(-used / float(outcomes)).sum()
    return float(-np.expm1(log_survival))

In [11]:
k = round((2 * 2**BITS * np.log(2)) ** 0.5 + 0.5)  # Round to nearest integer
model_p = collision_probability(k, 2**BITS)
observed_count = int(np.count_nonzero(waits <= k))

print(
    f"By draw {k}: model {model_p:.2%}, "
    f"observed {observed_count}/{TRIALS} "
    f"= {observed_count / TRIALS:.2%}"
)

By draw 302: model 50.07%, observed 469/900 = 52.11%


# Save the Trial Data for Analysis 
Trial data is saved in the `.csv` format which is compatible with R, SSPS, Pandas, Excel, and many more. 

The filename records width, trial count, and seed settings. 

In [12]:
def save_trials(dataset: list[dict]) -> Path:
    first = dataset[0]
    filename = f"trials_b{first['bits']}_n{len(dataset)}_seed{first['master_seed']}.csv"
    path = Path(filename)
    
    with path.open("w", newline="", encoding="utf-8") as stream:
        fields = ["trial", "trial_id", "bits", "master_seed", "T"]
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(dataset)
    return path

In [13]:
csv_path = save_trials(rows)
print(f"Saved {len(rows)} trials to {csv_path}")

Saved 900 trials to trials_b16_n900_seed37430101.csv


# Compare Widths

Keep the trial count and seed fixed, then add four bits at a time. 

Four additional bits give $2^4=16$ times as many labels. 

The birthday approximation $P(T>k)\approx\exp[-k(k-1)/(2M)]$ suggests a typical wait proportional to $\sqrt M$. 

That predicts roughly four times the wait for each step from 8 to 12 to 16 bits.

In [14]:
# Settings 
# ========
comparison_bits = (8, 12)

In [15]:
print("Bits      Labels    Median T      Mean T    Mean T / sqrt(M)")
for width in sorted({BITS, *comparison_bits}):
    if width == BITS:
        sample = waits
    else:
        other_rows = experiment_driver(TRIALS, width, MASTER_SEED)
        save_trials(other_rows)
        sample = np.array([row["T"] for row in other_rows])
    labels = 2**width
    print(f"{width:>4} {labels:>11,} {np.median(sample):>11g} "
          f"{sample.mean():>11.3f} {sample.mean() / labels**0.5:>19.3f}")

Bits      Labels    Median T      Mean T    Mean T / sqrt(M)
   8         256          19      20.478               1.280
  12       4,096          79      82.259               1.285
  16      65,536         294     315.814               1.234


# Interpreting Results with R

Open an R console in the folder that contains both the `.csv` files and `analysis.R`. 

To run the primary analysis in R: 

```R
source("analysis.R")
result <- analyze_trials("trials_b16_n900_seed37430101.csv")
```
